# kernelpilot on Google Colab (T4)
Runtime -> Change runtime type -> T4 GPU. The repo is private: create a GitHub token with read access and paste it when prompted, or make the repo public.

In [ ]:
import getpass, os
token = getpass.getpass('GitHub token (leave empty if the repo is public): ')
url = f'https://{token}@github.com/An1rudhSai/kernelpilot.git' if token else 'https://github.com/An1rudhSai/kernelpilot.git'
!git clone -q {url}
%cd kernelpilot

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q -e ".[dev]" "cmake>=3.24"

In [ ]:
# Build the HIP kernels for CUDA (sm_75 = T4) and run all correctness checks
!make build CUDA_ARCH=75
!make test

In [ ]:
# Benchmarks, markdown table and roofline plot
!make bench
from IPython.display import Image, Markdown, display
display(Markdown(open('results/bench.md').read()))
display(Image('results/roofline.png'))

In [ ]:
# Agent: any OpenAI-compatible endpoint
os.environ['KP_MODEL'] = 'your-model-name'
os.environ['KP_BASE_URL'] = 'https://api.openai.com/v1'
os.environ['KP_API_KEY'] = getpass.getpass('API key: ')
!kernelpilot optimize --kernel softmax --iterations 10

In [ ]:
!make evaluate
display(Markdown(open('results/evaluation.md').read()))